# CodeAlpha - Task 1: Credit Scoring Model
Predict creditworthiness using Logistic Regression, Decision Tree, and Random Forest.

## Cell 1: Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

print("Libraries imported successfully")

## Cell 2: Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload your CSV file here

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Shape:", df.shape)
df.head()

## Cell 2B (Optional): Sample Synthetic Dataset\nUse only if you don't have a real dataset. Skip if you uploaded one in Cell 2.

In [ ]:
# np.random.seed(42)
# n = 1000
# df = pd.DataFrame({
#     'income': np.random.normal(50000, 15000, n).clip(10000),
#     'debt': np.random.normal(15000, 8000, n).clip(0),
#     'age': np.random.randint(21, 65, n),
#     'employment_years': np.random.randint(0, 30, n),
#     'num_credit_lines': np.random.randint(0, 10, n),
#     'late_payments': np.random.poisson(1.5, n),
#     'credit_utilization': np.random.uniform(0, 1, n),
# })
# score = (
#     (df['income'] / 1000) - (df['debt'] / 1000)
#     - df['late_payments'] * 5
#     - df['credit_utilization'] * 20
#     + df['employment_years']
# )
# df['creditworthy'] = (score > score.median()).astype(int)
# df.head()

## Cell 3: Data Cleaning

In [ ]:
print("Missing values:\n", df.isnull().sum())
df = df.dropna()  # or df.fillna(df.median(numeric_only=True))

print("\nShape after cleaning:", df.shape)

## Cell 4: Feature Engineering

In [ ]:
if 'income' in df.columns and 'debt' in df.columns:
    df['debt_to_income'] = df['debt'] / (df['income'] + 1)

if 'late_payments' in df.columns:
    df['payment_risk_score'] = df['late_payments'] * 2

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

print("Feature engineering done")
df.head()

## Cell 5: Define Features (X) and Target (y)\nChange `target_col` to your actual target column name.

In [ ]:
target_col = 'creditworthy'

X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape, " y shape:", y.shape)

## Cell 6: Train-Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

## Cell 7: Train Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    }

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))

## Cell 8: Compare Model Performance

In [ ]:
results_df = pd.DataFrame(results).T
print("Model Comparison:")
print(results_df)

results_df.plot(kind='bar', figsize=(10, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

## Cell 9: ROC Curve (Random Forest)

In [ ]:
best_model = models["Random Forest"]
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_proba):.2f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig('roc_curve.png')
plt.show()

## Cell 10: Confusion Matrix (Random Forest)

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Creditworthy', 'Creditworthy'],
            yticklabels=['Not Creditworthy', 'Creditworthy'])
plt.title("Confusion Matrix - Random Forest")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.savefig('confusion_matrix.png')
plt.show()

## Cell 11: Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh')
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

## Cell 12: Save Model & Scaler

In [ ]:
import joblib
joblib.dump(best_model, 'credit_scoring_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model and scaler saved!")
print("\nBest Model Performance:")
print(results_df.loc["Random Forest"])

## Cell 13: Download Files

In [ ]:
files.download('credit_scoring_model.pkl')
files.download('scaler.pkl')